In [6]:
import pandas as pd
from sklearn.linear_model import LassoCV, Lasso, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_validate

df = pd.read_csv('AmesHousing.csv')
df = df.drop(columns=df.select_dtypes(include=object))
df = df.dropna()

X = df.drop(columns='SalePrice')
Y = df['SalePrice']

pipe = Pipeline([
      ('norm',StandardScaler()),
      ('lasso',LassoCV(cv=10, random_state=0))
])

pipe.fit(X, Y)

lamb_opm = pipe['lasso'].alpha_
print(f'Parâmetro de ajuste ótimo = {lamb_opm:.2f}')

Parâmetro de ajuste ótimo = 1091.68


In [4]:
model = Pipeline([
    ('norm',StandardScaler()),
    ('ridge',Lasso(alpha=lamb_opm))
])

cv = KFold(n_splits=10,shuffle=True,random_state=1)

results = cross_validate(model,X,Y,cv=cv,
                         scoring={'MSE':'neg_mean_squared_error',
                                  'RMSE':'neg_root_mean_squared_error',
                                  'MAE':'neg_mean_absolute_error',
                                  'R2':'r2'})

rmse = -results['test_RMSE']
r2 = results['test_R2']

print('Resultado via Lasso Regression:')
print(f'RMSE: {rmse.mean():.4f} ± {rmse.std():.4f}')
print(f'R²:   {r2.mean():.4f} ± {r2.std():.4f}')

### COMPARANDO COM LEAST SQUARES

model = Pipeline([
    ('OLS',LinearRegression())
])

cv = KFold(n_splits=10,shuffle=True,random_state=1)

results = cross_validate(model,X,Y,cv=cv,
                         scoring={'MSE':'neg_mean_squared_error',
                                  'RMSE':'neg_root_mean_squared_error',
                                  'MAE':'neg_mean_absolute_error',
                                  'R2':'r2'})

rmse = -results['test_RMSE']
r2 = results['test_R2']

print('\nResultado via Linear Regression:')
print(f'RMSE: {rmse.mean():.4f} ± {rmse.std():.4f}')
print(f'R²:   {r2.mean():.4f} ± {r2.std():.4f}')

Resultado via Lasso Regression:
RMSE: 35214.0927 ± 11947.8767
R²:   0.8085 ± 0.1326

Resultado via Linear Regression:
RMSE: 35413.4876 ± 12365.8708
R²:   0.8054 ± 0.1380


In [9]:
ols = LinearRegression()
ridge = Lasso(alpha=lamb_opm)

ols.fit(X, Y)
ridge.fit(X, Y)

for feature, b_ols, b_ridge in zip(X.columns, ols.coef_, ridge.coef_):

    print(f'{feature:20s} '
          f'OLS = {b_ols:10.4f}   '
          f'Lasso = {b_ridge:10.4f}')

Order                OLS =   -17.4104   Lasso =     0.8737
PID                  OLS =     0.0000   Lasso =    -0.0000
MS SubClass          OLS =  -185.0462   Lasso =  -198.5473
Lot Frontage         OLS =   -88.3415   Lasso =   -90.3858
Lot Area             OLS =     0.6338   Lasso =     0.6808
Overall Qual         OLS = 18078.4392   Lasso = 18929.5952
Overall Cond         OLS =  4190.1580   Lasso =  2251.4665
Year Built           OLS =   305.0382   Lasso =   280.4112
Year Remod/Add       OLS =   175.1357   Lasso =   299.3864
Mas Vnr Area         OLS =    34.8243   Lasso =    37.1435
BsmtFin SF 1         OLS =    13.0493   Lasso =    26.7654
BsmtFin SF 2         OLS =     3.0277   Lasso =    14.6354
Bsmt Unf SF          OLS =    -4.1262   Lasso =     4.6627
Total Bsmt SF        OLS =    11.9507   Lasso =     3.8610
1st Flr SF           OLS =    20.3091   Lasso =    49.0372
2nd Flr SF           OLS =    18.6695   Lasso =    49.7492
Low Qual Fin SF      OLS =    -4.2146   Lasso =    29.75